# 4. NetCDF Hazard Data Assessment

This notebook demonstrates damage assessment using NetCDF climate/hazard data (e.g., windstorms, climate projections).

In [ ]:
from pathlib import Path
import xarray as xr
from damagescanner import DamageScanner
import matplotlib.pyplot as plt

## Define input data

We use a windstorm NetCDF file with OSM building exposure data.

In [ ]:
data_path = Path("..") / "data" / "kampen"

hazard = data_path / "hazard" / "windstorm.nc"
exposure = data_path / "exposure" / "kampen.osm.pbf"
curves = data_path / "vulnerability" / "curves_osm.csv"
maxdam = data_path / "vulnerability" / "maxdam_osm.csv"

## Explore the NetCDF file

In [ ]:
nc = xr.open_dataset(hazard)
print("Variables:", list(nc.data_vars))
print("Dimensions:", dict(nc.sizes))
nc

## Initialize DamageScanner

In [ ]:
ds = DamageScanner(
    hazard_data=hazard,
    feature_data=exposure,
    curves=curves,
    maxdam=maxdam,
)

print(f"Assessment type: {ds.assessment_type}")
print(f"OSM mode: {ds.osm}")

## Exposure analysis for buildings

In [ ]:
exposed = ds.exposure(asset_type="buildings")

print(f"Exposed buildings: {len(exposed)}")
if len(exposed) > 0:
    print("Building types:")
    print(exposed["object_type"].value_counts())

## Calculate windstorm damages

In [ ]:
damages = ds.calculate(asset_type="buildings")

if len(damages) > 0:
    print(f"Buildings damaged: {len(damages)}")
    print(f"Total damage: €{damages['damage'].sum():,.0f}")
    print("Damage by building type:")
    print(damages.groupby("object_type")["damage"].sum().sort_values(ascending=False))
else:
    print("No damages calculated (buildings may be outside hazard extent)")

## Visualize hazard and exposure

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

# Get bounding box from exposed buildings (with buffer)
if len(exposed) > 0:
    bounds = exposed.total_bounds  # [minx, miny, maxx, maxy]
    buffer = 0.01  # Small buffer around buildings
    bbox = [
        bounds[0] - buffer,
        bounds[1] - buffer,
        bounds[2] + buffer,
        bounds[3] + buffer,
    ]
else:
    bbox = None

# Plot hazard - handle different NetCDF structures
nc_data = xr.open_dataset(hazard, engine="rasterio")

# Get the data variable
if "band_data" in nc_data:
    if "band" in nc_data.dims:
        plot_data = nc_data["band_data"].sel(band=1)
    elif "z" in nc_data.dims:
        plot_data = nc_data["band_data"].isel(z=0)
    else:
        plot_data = nc_data["band_data"]
else:
    var_name = list(nc_data.data_vars)[0]
    if "z" in nc_data.dims:
        plot_data = nc_data[var_name].isel(z=0)
    else:
        plot_data = nc_data[var_name]

# Clip to bbox if we have buildings
if bbox is not None:
    plot_data = plot_data.rio.clip_box(
        minx=bbox[0], miny=bbox[1], maxx=bbox[2], maxy=bbox[3]
    )

plot_data.plot(ax=ax, cmap="Reds", alpha=0.7)

# Overlay exposed buildings
if len(exposed) > 0:
    exposed.plot(ax=ax, color="blue", markersize=2, alpha=0.6)

# Set extent to bbox
if bbox is not None:
    ax.set_xlim(bbox[0], bbox[2])
    ax.set_ylim(bbox[1], bbox[3])

ax.set_title("Windstorm Hazard with Exposed Buildings")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.tight_layout()
plt.show()